In [1]:
from time import sleep, time
import threading

### sequential

In [2]:
def something():
    print("going to sleep...")
    sleep(1)
    print("woke up...")

start_time = time()

something()
something()
    
end_time = time()

print(f"Thread completed in {end_time-start_time} seconds")

going to sleep...
woke up...
going to sleep...
woke up...
Thread completed in 2.018108367919922 seconds


In [3]:
stat_time = time()
for _ in range(10):
    something()
end_time = time()

print(f"Thread completed in {end_time-start_time} seconds")

going to sleep...
woke up...
going to sleep...
woke up...
going to sleep...
woke up...
going to sleep...
woke up...
going to sleep...
woke up...
going to sleep...
woke up...
going to sleep...
woke up...
going to sleep...
woke up...
going to sleep...
woke up...
going to sleep...
woke up...
Thread completed in 12.13560938835144 seconds


### multi threading

In [4]:
def something(id):
    print(f"Thread {id} going to sleep...")
    sleep(2)
    print(f"Thread {id} woke up...")

In [5]:
start = time()

t1 = threading.Thread(target=something, args=[0])  #created thread
t1.start()

end = time()
print(f"\nThread ended in {end-start} seconds")

Thread 0 going to sleep...

Thread ended in 0.004815816879272461 seconds


### thread.join() - main thread will end after its children thread ends

In [6]:
start = time()

t1 = threading.Thread(target=something, args=[0])
t1.start()
t1.join()

end = time()
print(f"\nThread ended in {end-start} seconds")

Thread 0 going to sleep...
Thread 0 woke up...
Thread 0 woke up...

Thread ended in 2.0208728313446045 seconds


In [7]:
start = time()

t1 = threading.Thread(target=something, args=[0])
t2 = threading.Thread(target=something, args=[1])

t1.start() #thread 1 start
t2.start() #thread 2 start

#join both 
t1.join() 
t2.join()

end = time()
print(f"\nMain Thread ended in {end-start} seconds")

Thread 0 going to sleep...
Thread 1 going to sleep...
Thread 1 woke up...Thread 0 woke up...


Main Thread ended in 2.014082670211792 seconds


In [8]:
start = time()

threads = [threading.Thread(target=something, args=[i]) for i in range(10)]

for thread in threads:
    thread.start()
    
for thread in threads:
    thread.join()

end = time()
print(f"\nMain Thread ended in {end-start} seconds")

Thread 0 going to sleep...
Thread 1 going to sleep...
Thread 2 going to sleep...
Thread 3 going to sleep...
Thread 4 going to sleep...
Thread 5 going to sleep...
Thread 6 going to sleep...
Thread 7 going to sleep...
Thread 8 going to sleep...
Thread 9 going to sleep...
Thread 6 woke up...
Thread 9 woke up...
Thread 7 woke up...
Thread 5 woke up...
Thread 4 woke up...
Thread 3 woke up...
Thread 2 woke up...
Thread 1 woke up...
Thread 0 woke up...
Thread 8 woke up...

Main Thread ended in 2.0223793983459473 seconds


### locking threads

In [9]:
balance = 1000

def deposit(amt, times):
    global balance
    for _ in range(times):
        balance += amt
        
def withdraw(amt,times):
    global balance
    for _ in range(times):
        balance -= amt


In [10]:
deposit_thread = threading.Thread(target=deposit, args=[1,100000])
withdraw_thread = threading.Thread(target=withdraw, args=[1,100000])

deposit_thread.start()
withdraw_thread.start()

deposit_thread.join()
withdraw_thread.join()

print(balance)  
#wont be 1000 coz both threads updates balance at same time but system executes only one at a time

1000


### thread.lock() - will lock the thread
### thread.release() will release it after execution

In [11]:
balance = 1000

def deposit(amt, times,lock):
    global balance
    for _ in range(times):
        
        lock.acquire()
        balance += amt
        lock.release()
        
def withdraw(amt,times,lock):
    global balance
    for _ in range(times):
        
        lock.acquire()
        balance -= amt
        lock.release()

In [12]:
lock = threading.Lock()

deposit_thread = threading.Thread(target=deposit, args=[1,100000,lock])
withdraw_thread = threading.Thread(target=withdraw, args=[1,100000,lock])

deposit_thread.start()
withdraw_thread.start()

deposit_thread.join()
withdraw_thread.join()

print(balance)

1000


## ThreadPoolExecutor
### lets you run multiple tasks concurrently using a pool of threads. 

In [13]:
from concurrent.futures import ThreadPoolExecutor

In [14]:
def something(id):
    print(f"Thread {id} going to sleep...")
    sleep(2)
    return f"Thread {id} woke up..."

### single task 

In [15]:
start = time()

with ThreadPoolExecutor() as executor:
    task = executor.submit(something, 0)
    print(task.result())

end = time()
print(f"Time taken : {end - start}")

Thread 0 going to sleep...
Thread 0 woke up...
Time taken : 2.0132663249969482


### as_completed() - multiple tasks 

In [16]:
from concurrent.futures import ThreadPoolExecutor, as_completed

In [17]:
start = time()

with ThreadPoolExecutor() as executor:
    tasks = [executor.submit(something,i) for i in range(10)]  #no need to start and join the threads
    
    for completed_task in as_completed(tasks):
        print(completed_task.result())

end = time()
print(f"Time taken : {end-start}")

Thread 0 going to sleep...
Thread 1 going to sleep...
Thread 2 going to sleep...
Thread 3 going to sleep...
Thread 4 going to sleep...
Thread 5 going to sleep...
Thread 6 going to sleep...
Thread 7 going to sleep...
Thread 8 going to sleep...
Thread 9 going to sleep...
Thread 0 woke up...
Thread 1 woke up...
Thread 2 woke up...
Thread 8 woke up...
Thread 7 woke up...
Thread 6 woke up...
Thread 3 woke up...
Thread 9 woke up...
Thread 5 woke up...
Thread 4 woke up...
Time taken : 2.030172109603882


### printed as they complete

In [18]:
def work_something(id,time):
    print(f"Thread {id} going to sleep...")
    sleep(time)
    return f"Thread {id} woke up..."
    
start = time()

with ThreadPoolExecutor() as executor:
    tasks = [executor.submit(work_something,i,10-i) for i in range(10)]  #no need to start and join the threads
    
    for completed_task in as_completed(tasks):
        print(completed_task.result())

end = time()
print(f"Time taken : {end-start}")

Thread 0 going to sleep...
Thread 1 going to sleep...
Thread 2 going to sleep...
Thread 3 going to sleep...
Thread 4 going to sleep...
Thread 5 going to sleep...
Thread 6 going to sleep...
Thread 7 going to sleep...
Thread 8 going to sleep...
Thread 9 going to sleep...
Thread 9 woke up...
Thread 8 woke up...
Thread 7 woke up...
Thread 6 woke up...
Thread 5 woke up...
Thread 4 woke up...
Thread 3 woke up...
Thread 2 woke up...
Thread 1 woke up...
Thread 0 woke up...
Time taken : 10.021918058395386


### executor.map() - map the threads acc to id

In [20]:
ids = [5,2,3,1,4]

with ThreadPoolExecutor() as executor:
    results = executor.map(something,ids)
    
    for result in results:
        print(result)

Thread 5 going to sleep...
Thread 2 going to sleep...
Thread 3 going to sleep...
Thread 1 going to sleep...
Thread 4 going to sleep...
Thread 5 woke up...
Thread 2 woke up...
Thread 3 woke up...
Thread 1 woke up...
Thread 4 woke up...


### max_workers : controls how many threads can run at the same time in ThreadPoolExecutor

In [21]:
start = time()

with ThreadPoolExecutor(max_workers=3) as executor:
    tasks = [executor.submit(something,i) for i in range(10)]
    
    for completed_task in as_completed(tasks):
        print(completed_task.result())

end = time()
print(f"Time taken : {end-start}")

Thread 0 going to sleep...
Thread 1 going to sleep...
Thread 2 going to sleep...
Thread 3 going to sleep...Thread 1 woke up...

Thread 4 going to sleep...
Thread 5 going to sleep...
Thread 2 woke up...
Thread 0 woke up...
Thread 6 going to sleep...Thread 7 going to sleep...

Thread 8 going to sleep...
Thread 3 woke up...
Thread 5 woke up...
Thread 4 woke up...
Thread 9 going to sleep...Thread 7 woke up...
Thread 8 woke up...
Thread 6 woke up...

Thread 9 woke up...
Time taken : 8.042983293533325
